# 03 - Modelo M1: EfficientNet-B4 (piloto tecnico)

**Proposito del notebook:** Construir y entrenar M1 (EfficientNet-B4 con transferencia de aprendizaje desde ImageNet), siguiendo exactamente la arquitectura y el protocolo definidos en `configs/pilot_m1_m4.yaml`, sobre la particion real de ISIC 2019 generada en `02_preprocesamiento_y_particiones.ipynb`.

**Naturaleza de esta corrida:** esta es la ejecucion del **piloto tecnico**: una primera corrida con datos reales, arquitectura real y protocolo real, usando **train y validation solamente**. El conjunto de test **no se toca en este notebook**. El objetivo es validar que todo el pipeline funciona de punta a punta antes de congelar la configuracion y ejecutar las cinco corridas oficiales con distintas semillas.

**Entrada de datos:** `reports/splits/isic2019/isic2019_particion_70_15_15_seed42.csv`, generado y verificado en el notebook 02 (particion agrupada por lesion, sin fuga, estratificada por clase, semilla 42).

**Arquitectura (definida en `configs/pilot_m1_m4.yaml`, seccion `m1`):**

```text
Input: 380 x 380 x 3
        v
EfficientNetB4 backbone (include_top=False, weights=imagenet)
        v
GlobalAveragePooling2D
        v
BatchNormalization
        v
Dropout(0.30)
        v
Dense(256, ReLU, he_normal, L2=1e-4)
        v
Dropout(0.30)
        v
Dense(8, softmax, float32)

## Rutas portables, configuracion y particion

Se reutiliza el mismo mecanismo de localizacion de la raiz del proyecto de los notebooks anteriores. Se cargan `configs/pilot_m1_m4.yaml` (arquitectura y protocolo) y la particion persistida en el notebook 02, y se agrega `PROJECT_ROOT` a `sys.path` para poder importar `src.models.efficientnet_b4.build_efficientnet_b4_m1`, que construye M1 leyendo directamente la seccion `m1` de la configuracion.

In [1]:
import sys
from pathlib import Path

import yaml


def find_project_root(marker: str = "requirements.txt") -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(
        f"No se encontro '{marker}' en ningun directorio superior a {current}. "
        "Ejecuta este notebook desde dentro del repositorio del proyecto."
    )


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / "configs" / "pilot_m1_m4.yaml"
PARTICION_CSV = PROJECT_ROOT / "reports" / "splits" / "isic2019" / "isic2019_particion_70_15_15_seed42.csv"

CHECKPOINTS_DIR = PROJECT_ROOT / "checkpoints" / "m1_piloto"
LOGS_DIR = PROJECT_ROOT / "logs" / "m1_piloto"
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

print(f"Raiz del proyecto        : {PROJECT_ROOT}")
print(f"Configuracion cargada    : {CONFIG_PATH}")
print(f"Particion de entrada     : {PARTICION_CSV}")
print(f"Directorio de checkpoints: {CHECKPOINTS_DIR}")
print(f"Directorio de logs       : {LOGS_DIR}")
print(f"\nEstado de la configuracion (config['dataset']['partition']['materialization_status']): "
      f"{config['dataset']['partition']['materialization_status']}")

Raiz del proyecto        : /home/milat/proyecto_de_titulo
Configuracion cargada    : /home/milat/proyecto_de_titulo/configs/pilot_m1_m4.yaml
Particion de entrada     : /home/milat/proyecto_de_titulo/reports/splits/isic2019/isic2019_particion_70_15_15_seed42.csv
Directorio de checkpoints: /home/milat/proyecto_de_titulo/checkpoints/m1_piloto
Directorio de logs       : /home/milat/proyecto_de_titulo/logs/m1_piloto

Estado de la configuracion (config['dataset']['partition']['materialization_status']): pending_execution


## Carga de la particion

Se carga `isic2019_particion_70_15_15_seed42.csv` (generada en el notebook 02) y se separan explicitamente `train` y `validation`. El subconjunto `test` se carga solo para contar cuantas filas tiene (verificacion de integridad), pero **no se asigna a ninguna variable de trabajo** (no existe un `df_test`).

El protocolo del proyecto (`configs/pilot_m1_m4.yaml` -> `checkpoint_selection.test_set_used_for_selection: false`) prohibe usar el conjunto de prueba para cualquier decision antes de la evaluacion final (seleccion de checkpoint, ajuste de hiperparametros, comparacion de modelos). Si en una celda posterior de este notebook alguien escribiera por error codigo que use el test (por ejemplo, `df_test.head()`), Python levantaria inmediatamente un `NameError` en vez de ejecutar silenciosamente un uso indebido del test. Es decir: la ausencia de la variable actua como una barrera estructural contra la **fuga accidental de informacion del conjunto de prueba**.


In [2]:
import pandas as pd

df_particion = pd.read_csv(PARTICION_CSV)

conteo_splits = df_particion["split"].value_counts()
print("Filas por split (verificacion de integridad, incluye test solo para contar):")
print(conteo_splits)
print(f"\nTotal: {len(df_particion)} (esperado 25331)")

df_train = df_particion[df_particion["split"] == "train"].reset_index(drop=True)
df_val = df_particion[df_particion["split"] == "validation"].reset_index(drop=True)

print(f"\ndf_train: {len(df_train)} filas")
print(f"df_val  : {len(df_val)} filas")

print("\nDistribucion de clases en train:")
print(df_train["class"].value_counts().reindex(config["dataset"]["classes"]["order"]))
print("\nDistribucion de clases en validation:")
print(df_val["class"].value_counts().reindex(config["dataset"]["classes"]["order"]))

Filas por split (verificacion de integridad, incluye test solo para contar):
split
train         17705
validation     3818
test           3808
Name: count, dtype: int64

Total: 25331 (esperado 25331)

df_train: 17705 filas
df_val  : 3818 filas

Distribucion de clases en train:
class
AK       608
BCC     2304
BKL     1871
DF       174
MEL     3153
NV      8976
SCC      439
VASC     180
Name: count, dtype: int64

Distribucion de clases en validation:
class
AK       112
BCC      512
BKL      349
DF        36
MEL      721
NV      1948
SCC      100
VASC      40
Name: count, dtype: int64


## Pipeline de datos (tf.data) para M1

Se construye el pipeline de entrada usando `src/data/isic2019.py` (compartido con el futuro notebook 04 de M4, cambiando solo la normalizacion). Aplica, en orden: carga y resize bilinear a 380x380, aumento de datos solo en `train` (flip, rotacion, zoom isotropico, contraste, cada uno con probabilidad 0.5, segun `configs/pilot_m1_m4.yaml`), y la normalizacion especifica de M1 (identidad: EfficientNetB4 de Keras espera float32 en `[0, 255]` y aplica su propio reescalado interno, por lo que **no se divide por 255 aqui**).

`validation` no recibe aumento de datos, solo carga, resize y normalizacion, como exige el protocolo.

In [3]:
import tensorflow as tf

from src.data.isic2019 import build_dataset

BATCH_SIZE = config["common"]["batch_size"]
IMAGE_SIZE = config["m1"]["input_shape"][0]


def normalizar_m1(image):
    # EfficientNetB4 (Keras) espera float32 en [0, 255] y reescala internamente.
    return image


ds_train = build_dataset(
    df_train,
    PROJECT_ROOT,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    normalize_fn=normalizar_m1,
    training=True,
    num_classes=len(config["dataset"]["classes"]["order"]),
    augmentation_cfg=config["preprocessing"]["augmentation"],
    shuffle_seed=config["seed"],
)

ds_val = build_dataset(
    df_val,
    PROJECT_ROOT,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    normalize_fn=normalizar_m1,
    training=False,
    num_classes=len(config["dataset"]["classes"]["order"]),
)

for imagenes, etiquetas in ds_train.take(1):
    print(f"Batch de entrenamiento: {imagenes.shape}, dtype={imagenes.dtype}")
    print(f"Rango de pixeles: [{float(tf.reduce_min(imagenes))}, {float(tf.reduce_max(imagenes))}]")
    print(f"Etiquetas de ejemplo: {etiquetas.numpy()}")

for imagenes, etiquetas in ds_val.take(1):
    print(f"\nBatch de validacion: {imagenes.shape}, dtype={imagenes.dtype}")
    print(f"Etiquetas de ejemplo: {etiquetas.numpy()}")

2026-09-20 20:42:02.445173: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-09-20 20:42:03.600033: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
2026-09-20 20:42:06.154518: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-09-20 20:42:06.337446: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-09-20 20:42:06.337552: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] co

Batch de entrenamiento: (8, 380, 380, 3), dtype=<dtype: 'float32'>
Rango de pixeles: [0.0, 244.24722290039062]
Etiquetas de ejemplo: [[0. 0. 0. 0. 1. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1. 0. 0.]
 [0. 0. 1. 0. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 1. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0.]]

Batch de validacion: (8, 380, 380, 3), dtype=<dtype: 'float32'>
Etiquetas de ejemplo: [[0. 0. 0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1. 0. 0.]]


2026-09-20 20:42:09.293882: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2026-09-20 20:42:09.355394: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


## Construccion de M1

Se construye M1 usando `build_efficientnet_b4_m1` (definida en `src/models/efficientnet_b4.py`), que lee la arquitectura directamente desde la seccion `m1` de `configs/pilot_m1_m4.yaml`: backbone EfficientNetB4 con pesos ImageNet, congelado, mas la cabeza piloto completa (GAP -> BatchNormalization -> Dropout(0.30) -> Dense(256) -> Dropout(0.30) -> Dense(8, softmax)).

La funcion retorna tanto el modelo completo como una referencia directa al backbone, necesaria en la Fase 2 para descongelar selectivamente sus capas superiores sin afectar la cabeza.

In [4]:
from src.models.efficientnet_b4 import build_efficientnet_b4_m1

modelo_m1, backbone_m1 = build_efficientnet_b4_m1(config)

modelo_m1.summary(line_length=100)

print(f"\nBackbone congelado: {not backbone_m1.trainable}")
print(f"Parametros totales     : {modelo_m1.count_params():,}")
print(f"Parametros entrenables : {sum(tf.size(w).numpy() for w in modelo_m1.trainable_weights):,}")

Model: "efficientnet_b4_m1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                               ┃ Output Shape                    ┃           Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)                 │ (None, 380, 380, 3)             │                 0 │
├────────────────────────────────────────────┼─────────────────────────────────┼───────────────────┤
│ efficientnetb4 (Functional)                │ (None, 12, 12, 1792)            │        17,673,823 │
├────────────────────────────────────────────┼─────────────────────────────────┼───────────────────┤
│ global_average_pooling2d                   │ (None, 1792)                    │                 0 │
│ (GlobalAveragePooling2D)                   │                                 │                   │
├────────────────────────────────────────────┼─────────────────────────────────┼───────────────────┤
│ batch_normalization (BatchNormalization)   │ (None, 1792)                    │             7,168 │
├────────────────────────────────────────────┼─────────────────────────────────┼───────────────────┤
│ dropout (Dropout)                          │ (None, 1792)                    │                 0 │
├────────────────────────────────────────────┼─────────────────────────────────┼───────────────────┤
│ dense (Dense)                              │ (None, 256)                     │           459,008 │
├────────────────────────────────────────────┼─────────────────────────────────┼───────────────────┤
│ dropout_1 (Dropout)                        │ (None, 256)                     │                 0 │
├────────────────────────────────────────────┼─────────────────────────────────┼───────────────────┤
│ dense_1 (Dense)                            │ (None, 8)                       │             2,056 │
└────────────────────────────────────────────┴─────────────────────────────────┴───────────────────┘

 Total params: 18,142,055 (69.21 MB)

 Trainable params: 464,648 (1.77 MB)

 Non-trainable params: 17,677,407 (67.43 MB)


Backbone congelado: True
Parametros totales     : 18,142,055
Parametros entrenables : 464,648


## Pesos de clase y compilacion para Fase 1

Se calculan los pesos de clase segun la formula de `configs/pilot_m1_m4.yaml` (`w_c = N / (K * n_c)`, con `N` = total de imagenes en train, `K` = numero de clases, `n_c` = imagenes de la clase `c`), calculados **exclusivamente sobre `df_train`** (nunca sobre validation ni test).

Como `n_c` esta en el denominador, la relacion es inversa: clases con pocas imagenes (`DF`, `VASC`) reciben un peso alto, y la clase mayoritaria (`NV`) recibe un peso bajo. Esto compensa el desbalance porque `class_weight` multiplica la perdida de cada ejemplo por el peso de su clase durante el entrenamiento: sin este ajuste, el gradiente estaria dominado por los ~9.000 ejemplos de `NV` frente a los ~174 de `DF`, y el modelo podria minimizar la perdida promedio simplemente prediciendo la clase mayoritaria casi siempre, ignorando clases minoritarias que son clinicamente relevantes (`MEL`, `BCC`, `SCC`).

Se compila el modelo para la Fase 1: optimizador Adam nuevo (lr=1e-3, beta_1=0.9, beta_2=0.999, epsilon=1e-7), perdida `CategoricalCrossentropy` con `label_smoothing=0.05`, y como las etiquetas estan como enteros (`class_index`), se convierten a one-hot antes de calcular la perdida.

In [5]:
K = len(config["dataset"]["classes"]["order"])
N = len(df_train)

conteo_por_clase = df_train["class_index"].value_counts().sort_index()
pesos_clase = {idx: N / (K * n_c) for idx, n_c in conteo_por_clase.items()}

print("Pesos de clase (w_c = N / (K * n_c)), calculados solo desde train:")
for idx, peso in pesos_clase.items():
    nombre_clase = config["dataset"]["classes"]["order"][idx]
    print(f"  {nombre_clase} (idx={idx}): n_c={conteo_por_clase[idx]}, w_c={peso:.4f}")

opt_cfg = config["common"]["optimizer_defaults"]
optimizador_fase1 = tf.keras.optimizers.Adam(
    learning_rate=config["m1"]["training"]["phase_1"]["optimizer"]["learning_rate"],
    beta_1=opt_cfg["beta_1"],
    beta_2=opt_cfg["beta_2"],
    epsilon=opt_cfg["epsilon"],
)

perdida = tf.keras.losses.CategoricalCrossentropy(
    label_smoothing=config["common"]["loss"]["label_smoothing"]
)

modelo_m1.compile(
    optimizer=optimizador_fase1,
    loss=perdida,
    metrics=["accuracy"],
)

print(f"\nModelo compilado para Fase 1 (lr={config['m1']['training']['phase_1']['optimizer']['learning_rate']}).")

Pesos de clase (w_c = N / (K * n_c)), calculados solo desde train:
  AK (idx=0): n_c=608, w_c=3.6400
  BCC (idx=1): n_c=2304, w_c=0.9606
  BKL (idx=2): n_c=1871, w_c=1.1829
  DF (idx=3): n_c=174, w_c=12.7191
  MEL (idx=4): n_c=3153, w_c=0.7019
  NV (idx=5): n_c=8976, w_c=0.2466
  SCC (idx=6): n_c=439, w_c=5.0413
  VASC (idx=7): n_c=180, w_c=12.2951

Modelo compilado para Fase 1 (lr=0.001).


## Limitacion de hardware local y cambio de alcance

Al intentar ejecutar la Fase 1 completa (17.705 imagenes, hasta 15 epocas) en el equipo local, el sistema se apago espontaneamente durante la Epoca 3, consistente con una proteccion termica automatica del hardware (RTX 2060 en un equipo portatil, sin ventilacion pensada para cargas sostenidas de entrenamiento de deep learning).

En consecuencia, se reduce el alcance de este notebook en el equipo local a una **validacion mecanica del pipeline** (no un entrenamiento real): un subconjunto pequeno de imagenes, pocas epocas, sin pretender convergencia ni resultados interpretables. Su unico proposito es confirmar que el pipeline completo (carga de datos, aumento, modelo, callbacks, checkpoint) funciona de principio a fin sin errores, antes de ejecutar el piloto tecnico completo en el HPC Oceano PUCV, que si cuenta con hardware apto para sostener esta carga de forma segura.

Esta limitacion se documenta explicitamente porque es relevante: el piloto tecnico completo de M1 (y el entrenamiento de M4) quedan pendientes de ejecutarse en el HPC, no por una decision de alcance del proyecto, sino por una restriccion real de hardware local.

In [7]:
import random

random.seed(config["seed"])

# Muestra mas grande que antes (30 por clase en train, 10 en validation) para
# evitar un posible limite interno de Keras con datasets extremadamente chicos,
# manteniendo class_weight igual que en la corrida oficial.
df_train_muestra = (
    df_train.groupby("class", group_keys=False)
    .apply(lambda grupo: grupo.sample(n=min(30, len(grupo)), random_state=config["seed"]), include_groups=False)
    .reset_index(drop=True)
)
df_val_muestra = (
    df_val.groupby("class", group_keys=False)
    .apply(lambda grupo: grupo.sample(n=min(10, len(grupo)), random_state=config["seed"]), include_groups=False)
    .reset_index(drop=True)
)

print(f"Muestra de train para validacion de pipeline: {len(df_train_muestra)} imagenes")
print(f"Muestra de validation para validacion de pipeline: {len(df_val_muestra)} imagenes")

ds_train_muestra = build_dataset(
    df_train_muestra, PROJECT_ROOT, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
    normalize_fn=normalizar_m1, training=True, num_classes=K,
    augmentation_cfg=config["preprocessing"]["augmentation"], shuffle_seed=config["seed"],
)
ds_val_muestra = build_dataset(
    df_val_muestra, PROJECT_ROOT, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
    normalize_fn=normalizar_m1, training=False, num_classes=K,
)

# Modelo separado solo para esta validacion mecanica: no debe tocar ni
# sobrescribir `modelo_m1`, que la celda oficial de Fase 1 (mas abajo) usara
# para el entrenamiento real en el HPC.
modelo_validacion_pipeline, _ = build_efficientnet_b4_m1(config)
modelo_validacion_pipeline.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss=perdida, metrics=["accuracy"])

historia_validacion_pipeline = modelo_validacion_pipeline.fit(
    ds_train_muestra,
    validation_data=ds_val_muestra,
    epochs=2,
    class_weight=pesos_clase,
)

print("\nPipeline validado de principio a fin (carga, augmentation, modelo, entrenamiento, class_weight).")
print("Esto NO es un resultado de desempeno real: usa un subconjunto minimo sin intencion de convergencia.")

Muestra de train para validacion de pipeline: 240 imagenes
Muestra de validation para validacion de pipeline: 80 imagenes
Epoch 1/2


2026-09-20 20:46:50.716387: W tensorflow/core/kernels/data/prefetch_autotuner.cc:52] Prefetch autotuner tried to allocate 16777472 bytes after encountering the first element of size 16777472 bytes.This already causes the autotune ram budget to be exceeded. To stay within the ram budget, either increase the ram budget or reduce element size
2026-09-20 20:46:50.717338: W tensorflow/core/kernels/data/prefetch_autotuner.cc:52] Prefetch autotuner tried to allocate 16777504 bytes after encountering the first element of size 16777504 bytes.This already causes the autotune ram budget to be exceeded. To stay within the ram budget, either increase the ram budget or reduce element size


30/30 ━━━━━━━━━━━━━━━━━━━━ 91s 382ms/step - accuracy: 0.2667 - loss: 11.7672 - val_accuracy: 0.2125 - val_loss: 2.1533
Epoch 2/2
 2/30 ━━━━━━━━━━━━━━━━━━━━ 2s 100ms/step - accuracy: 0.0312 - loss: 6.2262    

2026-09-20 20:48:01.228374: W tensorflow/core/kernels/data/prefetch_autotuner.cc:52] Prefetch autotuner tried to allocate 16777472 bytes after encountering the first element of size 16777472 bytes.This already causes the autotune ram budget to be exceeded. To stay within the ram budget, either increase the ram budget or reduce element size
2026-09-20 20:48:01.235569: W tensorflow/core/kernels/data/prefetch_autotuner.cc:52] Prefetch autotuner tried to allocate 16777504 bytes after encountering the first element of size 16777504 bytes.This already causes the autotune ram budget to be exceeded. To stay within the ram budget, either increase the ram budget or reduce element size


30/30 ━━━━━━━━━━━━━━━━━━━━ 3s 108ms/step - accuracy: 0.3042 - loss: 7.7625 - val_accuracy: 0.2500 - val_loss: 2.0921

Pipeline validado de principio a fin (carga, augmentation, modelo, entrenamiento, class_weight).
Esto NO es un resultado de desempeno real: usa un subconjunto minimo sin intencion de convergencia.


## Entrenamiento Fase 1 (cabeza, backbone congelado)

Se entrena solo la cabeza de M1 (backbone completamente congelado, incluido su BatchNormalization interno) durante hasta 15 epocas, con los callbacks definidos en `configs/pilot_m1_m4.yaml`:

- `EarlyStopping` sobre `val_loss` (patience=7, min_delta=1e-4, restore_best_weights=True).
- `ReduceLROnPlateau` sobre `val_loss` (factor=0.2, patience=3, min_lr=1e-7).
- `ModelCheckpoint` guardando el mejor `val_loss` de esta fase en `checkpoints/m1_piloto/fase1_mejor.weights.h5`.

Los pesos de clase calculados en el paso anterior se pasan a `model.fit(..., class_weight=...)`, aplicandose solo durante el entrenamiento (nunca sobre la evaluacion de validation).

## Nota: la celda de entrenamiento completo que sigue es para el HPC, no para el equipo local

La celda de codigo bajo "Entrenamiento Fase 1" (mas abajo) contiene el codigo real y correcto de la Fase 1 completa, con `ds_train`/`ds_val` completos (17.705 / 3.818 imagenes) y `class_weight`. **No se debe ejecutar en este equipo local**: el primer intento causo un apagon por proteccion termica durante la Epoca 3 (ver seccion anterior, "Limitacion de hardware local").

Este codigo queda documentado y listo para ejecutarse sin cambios en el HPC Oceano PUCV, donde el hardware si esta pensado para sostener esta carga de forma segura.

In [ ]:
cb_cfg = config["common"]["callbacks"]

callbacks_fase1 = [
    tf.keras.callbacks.EarlyStopping(
        monitor=cb_cfg["early_stopping"]["monitor"],
        mode=cb_cfg["early_stopping"]["mode"],
        patience=cb_cfg["early_stopping"]["patience"],
        min_delta=cb_cfg["early_stopping"]["min_delta"],
        restore_best_weights=cb_cfg["early_stopping"]["restore_best_weights"],
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor=cb_cfg["reduce_lr_on_plateau"]["monitor"],
        mode=cb_cfg["reduce_lr_on_plateau"]["mode"],
        factor=cb_cfg["reduce_lr_on_plateau"]["factor"],
        patience=cb_cfg["reduce_lr_on_plateau"]["patience"],
        min_lr=cb_cfg["reduce_lr_on_plateau"]["min_lr"],
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(CHECKPOINTS_DIR / "fase1_mejor.weights.h5"),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        save_weights_only=True,
    ),
]

historia_fase1 = modelo_m1.fit(
    ds_train,
    validation_data=ds_val,
    epochs=config["m1"]["training"]["phase_1"]["max_epochs"],
    class_weight=pesos_clase,
    callbacks=callbacks_fase1,
)